# Depth Anything V2 Small — DIMER relative depth estimation tutorial

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/depth-anything-depth-estimation-pipeline)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/depth-anything-depth-estimation-pipeline/blob/main/tutorials/depth_anything_depth_estimation_colab.ipynb)
[![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-depth--anything%2FDepth--Anything--V2--Small--hf-ffcc4d?style=flat)](https://huggingface.co/depth-anything/Depth-Anything-V2-Small-hf)
[![Upstream](https://img.shields.io/badge/Upstream-DepthAnything%2FDepth--Anything--V2-181717?style=flat&logo=github&logoColor=white)](https://github.com/DepthAnything/Depth-Anything-V2)
[![arXiv](https://img.shields.io/badge/arXiv-2406.09414-b31b1b.svg)](https://arxiv.org/abs/2406.09414)

**Profile:** `TASK-INFERENCE`  
**Notebook specification:** DIMER Notebook Specification 1.0  
**Capability:** monocular relative depth estimation from one RGB still image using the pinned Depth Anything V2 Small weights

This notebook is the executable reference path for the repository capability. It exercises the repository's public pipeline API (`DepthAnythingPipeline`) rather than reimplementing model inference. The output is a per-pixel map of **relative inverse depth** — larger values are nearer, the scale and shift are unknown, and the values are **not metric**: they are not distances in metres and cannot be compared across images without an alignment step. No training, fine-tuning, in-context conditioning, or preprocessing fitting occurs; the pinned checkpoint is used as published, and the only state the repository adds is manifest verification, input validation, and resizing of the output back to the input resolution.

**Learning objectives:** bootstrap the repository in a fresh runtime, generate a synthetic default input (or upload your own), validate it against the pipeline's enforced ceilings, stage and digest-verify the immutable upstream snapshot, run the supported task through the public API, interpret the relative-depth output and its sanity checks, understand when the shipped `abs_rel` metric applies and why no metric is reported here, and export machine-readable outputs plus provenance.

**This notebook does not demonstrate:** metric (absolute) depth, video or temporal depth, stereo or multi-view fusion, surface normals, 3D reconstruction, or batch inference. The repository does not provide metric depth, video, or batched paths, and this notebook must not be read as implying them.

## Prerequisites

Run in a fresh supported runtime. The default path supports CPU and uses CUDA automatically when available; the model is about 99 MB and the default sample is 320 x 240 px, so a CPU runtime completes the default path (the model card records a 320 x 240 CPU prediction at 0.34 s cold on the card's workstation; a hosted CPU runtime may be slower and no figure is claimed for it). Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded images remain in the notebook runtime; this pipeline does not send them to a third-party inference API. The only network access on the default path is the Git clone, the pinned wheel installs, and the fetch of the model files from the Hugging Face Hub at the immutable revision.

Knowledge assumed: basic Python and NumPy, and the difference between inverse depth (larger = nearer, arbitrary scale and shift) and metric distance.

## 1. Bootstrap the repository and pinned runtime

When the notebook is opened without a repository checkout, this cell clones the repository. Released notebooks default to `main`; automated candidate validation can set `DIMER_TUTORIAL_REF` to an immutable commit or review branch. The repository is installed as a regular (non-editable) package so it is importable in this same runtime; an editable install would only become importable after a restart. Model-facing dependencies (`torch`, `transformers`, `safetensors`, `numpy`, `pillow`) are directly pinned by `pyproject.toml`. If installation replaces any package that this runtime has already imported (hosted runtimes commonly pre-import a different NumPy or Pillow), the cell fails with a restart instruction rather than continuing with mixed versions: restart the runtime and rerun from the top. Inference runs in float32 on CUDA when available, otherwise on CPU; no half precision, compilation, or quantization is applied.

In [ ]:
import importlib
import importlib.metadata
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/kurtvalcorza/depth-anything-depth-estimation-pipeline.git'
REPO_NAME = 'depth-anything-depth-estimation-pipeline'
REPO_REF = os.environ.get('DIMER_TUTORIAL_REF', 'main')
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    checkout = ROOT / REPO_NAME
    if not checkout.exists():
        subprocess.run(['git', 'clone', '--filter=blob:none', '-q', REPO_URL, str(checkout)], check=True)
    if REPO_REF != 'main':
        subprocess.run(['git', '-C', str(checkout), 'fetch', '--depth', '1', 'origin', REPO_REF], check=True)
        subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
    else:
        subprocess.run(['git', '-C', str(checkout), 'checkout', '-q', 'main'], check=True)
        subprocess.run(['git', '-C', str(checkout), 'pull', '--ff-only', '-q', 'origin', 'main'], check=True)
    os.chdir(checkout)
    ROOT = Path.cwd()

if not SKIP_INSTALL:
    # Every distribution that is already imported in this runtime is captured before installation,
    # whatever its name (PIL -> pillow), so a pinned install that replaces any loaded package is
    # detected. Distribution metadata is compared with metadata afterwards: torch.__version__ carries
    # a local build label (for example 2.14.0+cu130) that the distribution version omits.
    def _installed_version(distribution):
        try:
            return importlib.metadata.version(distribution)
        except importlib.metadata.PackageNotFoundError:
            return None
    _module_dists = importlib.metadata.packages_distributions()
    _loaded_dists = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded_dists}
    # Non-editable install: an editable (.pth) install is not importable until the
    # interpreter restarts, which a fresh hosted runtime cannot do mid-notebook.
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', str(ROOT)], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

REPO_SHA = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
import platform, PIL, numpy, torch, transformers
print({'repository': str(ROOT), 'repository_revision': REPO_SHA, 'requested_ref': REPO_REF, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'numpy': numpy.__version__, 'pillow': PIL.__version__, 'cuda': torch.cuda.is_available()})

## 2. Generate the synthetic sample or optional BYOD

The default sample is **synthetic**: a 320 x 240 RGB image drawn in this cell — a left-to-right luminance ramp with a bright square and a dark disc — the same kind of input the repository's smoke run used. It is generated deterministically from code (no randomness, so no seed is involved), its pixel digest is recorded in the export so a rerun can prove it saw the same input, and it needs no download and contains no personal data. It ships **no ground-truth depth**, and as a non-photographic image it lies outside the model's training distribution, so the depth map it produces is smoke/sanity evidence of the code path only — it says nothing about depth quality on real photographs and is not benchmark evidence.

BYOD is optional and disabled by default. Expected BYOD input: one image file that Pillow can open (PNG, JPEG, WebP, …), any mode (it is converted to RGB), with shorter side at least `MIN_IMAGE_SIDE` = 14 px, longer side at most `MAX_IMAGE_SIDE` = 4096 px, and aspect ratio at most `MAX_ASPECT_RATIO` = 4.0 — the next section checks these before the model runs. The upload stays inside this runtime. If you also hold a metric depth map for your image (a float array in metres with the same height and width, from a depth sensor or a benchmark), assign it to `reference_depth` after the upload and Section 5 will score it with the repository's `abs_rel` helper; leave it as `None` otherwise.

In [ ]:
import hashlib
import io

import numpy as np
from PIL import Image, ImageDraw

USE_BYOD = False  # @param {type:"boolean"}
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    sample_name = next(iter(uploaded))
    image = Image.open(io.BytesIO(uploaded[sample_name]))
    image.load()
    sample_kind = 'BYOD upload'
else:
    # Deterministic synthetic scene: a left-to-right luminance ramp (40 -> 220) with a bright
    # square and a dark disc. Drawn from code, so it is reproducible without any download.
    width, height = 320, 240
    ramp = np.linspace(40, 220, width, dtype=np.float32)
    rgb = np.repeat(np.repeat(ramp[None, :, None], height, axis=0), 3, axis=2).astype(np.uint8)
    image = Image.fromarray(rgb)
    draw = ImageDraw.Draw(image)
    draw.rectangle([200, 60, 280, 140], fill=(245, 245, 245))
    draw.ellipse([40, 120, 130, 210], fill=(20, 20, 20))
    sample_name = 'synthetic_ramp_320x240'
    sample_kind = 'synthetic (generated in this cell)'
# Metric ground-truth depth in metres with the same H x W as the image, or None. The synthetic
# sample has none, so no metric is computed for it.
reference_depth = None
sample_sha256 = hashlib.sha256(np.asarray(image.convert('RGB')).tobytes()).hexdigest()
print({'sample': sample_name, 'sample_kind': sample_kind, 'mode': image.mode, 'size': image.size, 'has_reference_depth': reference_depth is not None, 'pixel_sha256': sample_sha256})

## 3. Validate the input against the pipeline ceilings

The pipeline enforces three operational ceilings, imported here from the package so the values shown are the ones in force: `MIN_IMAGE_SIDE` (shorter side, px), `MAX_IMAGE_SIDE` (longer side, px) and `MAX_ASPECT_RATIO` (longer / shorter). This cell surfaces them and checks the input before any model work, naming the failing condition and the corrective action; `predict()` re-applies the same rules authoritatively and raises `TypeError`/`ValueError` on its own. The notebook does not crop, resize, or subsample the input. Inside the pipeline the image processor rescales the image so its sides are multiples of 14 near 518 px and the raw prediction is interpolated back to the input resolution, so detail finer than that internal scale is lost regardless of the source resolution; the model card records that peak accelerator memory grows with aspect ratio (812 MiB at 4096 x 1024 versus 298 MiB at 4096 x 4096 on the card's GPU), which is why an aspect ceiling exists.

In [ ]:
from depth_anything_depth_estimation_pipeline import MAX_ASPECT_RATIO, MAX_IMAGE_SIDE, MIN_IMAGE_SIDE

width, height = image.size
short_side, long_side = min(width, height), max(width, height)
ceilings = {'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_ASPECT_RATIO': MAX_ASPECT_RATIO}
print(ceilings)
problems = []
if short_side < MIN_IMAGE_SIDE:
    problems.append(f'shorter side {short_side} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}: supply a larger image')
if long_side > MAX_IMAGE_SIDE:
    problems.append(f'longer side {long_side} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}: downscale the image before calling predict')
if long_side / short_side > MAX_ASPECT_RATIO:
    problems.append(f'aspect ratio {long_side / short_side:.2f} > MAX_ASPECT_RATIO {MAX_ASPECT_RATIO}: crop to a squarer frame')
if problems:
    raise ValueError('input rejected before model execution: ' + '; '.join(problems))
print({'width': width, 'height': height, 'aspect_ratio': round(long_side / short_side, 3), 'within_ceilings': True})

## 4. Stage, verify, and resolve the pinned model

The public API pins the exact upstream model repository and immutable 40-hex revision and refuses remote model code (`trust_remote_code=False`). The repository commits the DIMER snapshot manifest (`weights/depth-anything-v2-small/dimer-base-manifest.json`: model id, revision, and the byte size and SHA-256 of each of the four snapshot files) but git-ignores the 99 MB `model.safetensors`, so a fresh clone must stage the missing files first. The package's `stage_missing_files(WEIGHTS_DIR, allow_download=True)` fetches only the manifest-listed files that are absent, from the Hub at the pinned revision, into the repository's weights directory, and returns the list it fetched (`['model.safetensors']` on a fresh clone, `[]` when everything is already staged); it refuses to stage if the committed manifest disagrees with the package's pinned identity, and it never touches the Hub without the explicit flag. `verify_snapshot()` then checks every listed file's size and digest against the manifest and raises on the first mismatch, and only afterwards does `from_pretrained` load the model from that verified directory with `local_files_only=True` — the loader never silently falls back to a different download. The effective model identity, the snapshot summary, and the selected device are printed before inference. Look for the `verify_snapshot` summary reporting `files: 4` and the pinned revision.

In [ ]:
import json

from depth_anything_depth_estimation_pipeline import MODEL_ID, MODEL_KEY, MODEL_REVISION, DepthAnythingPipeline, abs_rel, stage_missing_files, verify_snapshot

print({'model_id': MODEL_ID, 'revision': MODEL_REVISION})
WEIGHTS_DIR = ROOT / 'weights' / MODEL_KEY
# Only the manifest-listed files that are absent are fetched, at the immutable revision the
# package pins; verify_snapshot then checks every byte count and SHA-256 before anything is loaded.
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'fetched': fetched, 'from': MODEL_ID, 'revision': MODEL_REVISION})
snapshot_info = verify_snapshot(WEIGHTS_DIR)
print(snapshot_info)
pipe = DepthAnythingPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': pipe.device, 'precision': 'float32'})

## 5. Run relative depth estimation and evaluate when ground truth exists

`predict()` returns a dictionary: `depth` (float32 `H x W` at the input resolution), `depth_kind` (`"relative"`), `depth_min`, `depth_max`, `height`, `width`, `model_id`, `model_revision`. **Score semantics:** each value is relative inverse depth — larger means nearer — with an unknown per-image scale and shift. The values are not distances, not probabilities, and carry no confidence map; the pipeline applies no decision threshold, no binarisation, and no unit conversion, and any near/far cut-off is owned by the downstream caller. The sanity checks below are falsifiable plumbing checks (shape equals the input, float32, finite, non-degenerate range); they are not a quality measure.

**Evaluation:** the only metric the repository ships is `abs_rel(pred, ref_depth, align=True)` — absolute relative error `mean(|est - ref| / ref)` over pixels with `ref_depth > 0`, computed after the prediction is affinely aligned to `1 / ref_depth` by least squares and inverted to depth. It applies only when the caller supplies metric ground-truth depth of the same height and width. The synthetic sample has none, so **no metric is reported** on the default path and the cell prints that fact instead of a number; a meaningful trivial baseline (for example a constant or vertical-gradient depth prior) likewise needs ground truth to be scored against, so none is reported either. To evaluate on your own data you need a metric depth map per image (from a depth sensor, LiDAR, or an RGB-D benchmark), and a single-image `abs_rel` is tutorial evidence with no dispersion estimate, not a benchmark result. The runtime figure printed below is measured on the runtime identified in Section 1 for this one image.

In [ ]:
import time

started = time.perf_counter()
result = pipe.predict(image)
elapsed = time.perf_counter() - started
depth = result['depth']
checks = {
    'shape_matches_input': depth.shape == (result['height'], result['width']) == (image.height, image.width),
    'dtype_float32': depth.dtype == np.float32,
    'all_finite': bool(np.isfinite(depth).all()),
    'non_degenerate_range': result['depth_max'] > result['depth_min'],
}
if not all(checks.values()):
    raise RuntimeError(f'depth map failed a sanity check: {checks}')
print({key: value for key, value in result.items() if key != 'depth'})
print({'depth_shape': depth.shape, 'seconds': round(elapsed, 3), 'checks': checks})
metrics = {}
if reference_depth is not None:
    metrics['abs_rel'] = abs_rel(depth, reference_depth, align=True)
    print({'abs_rel': metrics['abs_rel'], 'estimation': 'single image, affine-aligned in inverse depth; tutorial evidence only'})
else:
    print('no metric is reported: the sample ships no ground-truth depth, so abs_rel is not computed')

## 6. Visualize the relative depth map

The preview is a per-image min–max stretch of the relative inverse depth to 8-bit grey, shown beside the input: brighter means nearer (a larger value). The grey levels are a visual aid only — they are not distances, the stretch discards the model's scale, and two previews cannot be compared with each other. The machine-readable arrays exported in the next section, not this picture, are the outputs intended for downstream use. On the synthetic sample expect a plausible but meaningless map: the input is not a photograph.

In [ ]:
lo, hi = result['depth_min'], result['depth_max']
depth_u8 = np.round((depth - lo) / (hi - lo) * 255.0).astype(np.uint8)
depth_preview = Image.fromarray(depth_u8).convert('RGB')
side_by_side = Image.new('RGB', (image.width * 2, image.height))
side_by_side.paste(image.convert('RGB'), (0, 0))
side_by_side.paste(depth_preview, (image.width, 0))
try:
    from IPython.display import display
    display(side_by_side)
except ImportError:
    print({'preview': 'IPython display unavailable; the preview PNG is written in the next section'})

## 7. Export outputs and provenance

Three files are written under `outputs/`: the full-resolution float32 depth array (`.npy`, the array intended for downstream use), the side-by-side preview PNG, and a JSON record that ties them to the sample identity (name, kind, pixel digest, size), the depth summary (`depth_kind`, min, max, shape, dtype, measured seconds), the metric block (empty when no ground truth was supplied), the sanity checks, the ceilings in force, the repository revision, the model identifier and immutable revision, the verified snapshot summary, and the runtime identity (Python, PyTorch, Transformers, NumPy, Pillow, device, precision). No credentials are involved in any step, so none can reach the export.

In [ ]:
os.makedirs('outputs', exist_ok=True)
np.save('outputs/depth_anything_depth_estimation_depth.npy', depth)
side_by_side.save('outputs/depth_anything_depth_estimation_preview.png')
payload = {
    'sample': {
        'name': sample_name,
        'kind': sample_kind,
        'pixel_sha256': sample_sha256,
        'width': image.width,
        'height': image.height,
        'has_reference_depth': reference_depth is not None,
    },
    'prediction': {
        'depth_file': 'outputs/depth_anything_depth_estimation_depth.npy',
        'preview_file': 'outputs/depth_anything_depth_estimation_preview.png',
        'depth_kind': result['depth_kind'],
        'depth_min': result['depth_min'],
        'depth_max': result['depth_max'],
        'shape': list(depth.shape),
        'dtype': str(depth.dtype),
        'seconds': round(elapsed, 3),
    },
    'metrics': metrics,
    'sanity_checks': checks,
    'ceilings': ceilings,
    'repository_revision': REPO_SHA,
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'snapshot': snapshot_info,
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'numpy': numpy.__version__,
        'pillow': PIL.__version__,
        'device': pipe.device,
        'precision': 'float32',
    },
}
with open('outputs/depth_anything_depth_estimation_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))
print('outputs/depth_anything_depth_estimation_result.json')

## Interpretation and limits

The depth map is a model-generated map of relative inverse depth: larger is nearer, the scale and shift are unknown, and the values are not metres, not probabilities, and not comparable across images without alignment. On the synthetic default sample the map has no ground truth and the input is not a photograph, so the run demonstrates the code path and its outputs, not depth quality; no metric is reported because none can be computed without caller-supplied metric depth, and where `abs_rel` is computed on your own image it is a single-image tutorial figure with no dispersion estimate. The pipeline provides no confidence map, no calibrated threshold, and no metric conversion; it does not provide metric depth, video or temporal consistency, stereo/multi-view fusion, normals, 3D reconstruction, or batching. Depth quality on mirrors, glass, textureless surfaces, night scenes, fog, and non-photographic inputs is expected to degrade and is not signalled. Run-to-run variability after the deterministic sample comes from floating-point kernel selection across devices (the model card records `depth_max` 2.969 on CUDA versus 2.970 on CPU for its smoke image) and from bicubic interpolation; results on fixed hardware are repeatable but not guaranteed bitwise-identical across devices.

Successful execution proves that the recorded repository revision can bootstrap in a fresh runtime, stage and digest-verify the pinned model snapshot, validate the demonstrated input against the enforced ceilings, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime. It does **not** establish benchmark superiority, deployment calibration, accuracy on any real-image domain, safety for high-consequence decisions, or production fitness on an unseen domain.

**Troubleshooting.** `RuntimeError: Core dependencies changed while older modules were loaded` in Section 1: the pinned install replaced a package the runtime had pre-imported — restart the runtime and rerun from the top. `FileNotFoundError: snapshot file missing` or a `sha256`/`size` `ValueError` in Section 4: a staged file is incomplete or altered — delete it from `weights/depth-anything-v2-small/` and rerun Section 4. `ValueError` naming `MIN_IMAGE_SIDE`, `MAX_IMAGE_SIDE` or `MAX_ASPECT_RATIO` in Section 3 or 5: the BYOD image is outside the ceilings — resize or crop it. An out-of-memory error on a near-ceiling BYOD image: the card measured 812 MiB peak at 4096 x 1024 on its GPU; use a smaller or squarer image or a CPU runtime.

**Next experiments.** Upload a real photograph with `USE_BYOD` enabled and inspect whether the near/far ordering matches the scene; if you hold a metric depth map for it, assign it to `reference_depth` and compare `abs_rel` with `align=True` against `align=False` to see why affine alignment is required; compare the CUDA and CPU `depth_min`/`depth_max` on the same image to observe kernel-level variability. None of these turns the sample result into evidence of production fitness.

## References

- Repository README: `../README.md`
- Repository model card: `../MODEL_CARD.md`
- Weight provenance: `../docs/WEIGHTS.md`
- Upstream model: https://huggingface.co/depth-anything/Depth-Anything-V2-Small-hf
- Upstream code: https://github.com/DepthAnything/Depth-Anything-V2
- Depth Anything V2 paper: https://arxiv.org/abs/2406.09414
- Depth Anything (V1) paper: https://arxiv.org/abs/2401.10891
- Transformers `DepthAnything` documentation: https://huggingface.co/docs/transformers/model_doc/depth_anything